# Exploring Corpus

This notebook explores and describes the corpus. This is done to verify if paper processing into the corpus had gone correctly. (quality of data)

- Frequency of header names (What is the variation in header names?)
- What is the frequency of words in sentences (group Parkinson vs non-Parkinson)?
- How many words are there in a sentence?
- How many sentences are there in a paper?

## Initialisation

In [ ]:
# meta
__author__ ="Jennefer Beenen"
__version__ = "1.0"
__email__ = "j.beenen@pl.hanze.nl"
__status__ = "Development"
# __date__ = "2025-05-06"

In [ ]:
# imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# https://amueller.github.io/word_cloud/
from wordcloud import WordCloud
from wordcloud import STOPWORDS

from sentence_transformers import SentenceTransformer

### Settings

In [ ]:
# load data
filename = 'corpus_free_1000_251013_pest_PD'
df = pd.read_csv(f'../../data/corpus/{filename}.csv')
df.info()

### Functions

In [ ]:
def unique_values_per_column(df: pd.DataFrame):
    """Prints the unique values found in each column of a data frame."""
    for column in df.columns:
        if df[column].nunique() < 25:
            print(f"'{column}' ({df[column].nunique()}):\n{df[column].unique()}\n")
        else:
            print(f"'{column}' ({df[column].nunique()}): (only first 25 values are shown)\n{df[column].unique()[:25]}\n")

def df_describe_not_norm(series):
    """Return dataframe with describing values for a non-normal distribution."""
    # create dataframe with median, min, and max
    new_df = pd.DataFrame.from_dict({
            "count": series.shape[0],
            "median": series.median(),
            "min": series.min(),
            "max": series.max()
    }, orient= 'index').T

    # add value range
    new_df['range'] = new_df['max'] - new_df['min']

    return new_df.T

def get_sentences(series, keywords: list, inverse: bool = False) -> list:
    """Get sentences from series that contains the keyword, or sentences without the keyword by setting `inverse` to `True`."""
    # Make all sentence_text lower case & remove puntuations
    # https://www.geeksforgeeks.org/python/generating-word-cloud-python/
    series = series.str.lower().str.strip('.?!')

    if inverse:
        # Search for sentences that exclude `keywords`
        # Note: Copilot brought to my attention that `series.str.contains` exists
        # and that it can handle a list by creating a regex expression with "|".join()
        sentences = series.loc[~series.str.contains("|".join(keywords))]
        # Report number of sentences found
        print(f"{len(sentences)} sentences were found without '{keywords}'.")   
    else:
        # Search for sentences that include `keywords`
        sentences = series.loc[series.str.contains("|".join(keywords))]
        # Report number of sentences found
        print(f"{len(sentences)} sentences were found with '{keywords}'.")

    # (Ready to create wordcloud)
    return sentences

def create_wordcloud(sentences: list):
    """Create WordCloud object, 16:9 ratio, top 100 words."""
    # Create WordCloud object, 16:9 ratio, top 100 words
    wordcloud = WordCloud(
        width=1600,
        height=900,
        max_words= 100,
        stopwords= STOPWORDS.update(['one', 'two', 'three', 'first', 'study']),
        colormap= 'PiYG'
    ).generate(" ".join(sentences)) # join words to form one long 'sentence'

    # Plot
    plt.imshow(wordcloud)
    plt.axis('off')
    plt.show()

### Replace 'PD' with "parkinson's disease"

I'm expecting Parkinson's Disease to be often abbrevated to PD.

In [ ]:
# Current state:
df.head()

In [ ]:
# Check if "PD" can mean Parkinson.
PD_indexes = df['sentence_text'].str.findall("PD").str.len() > 0
# Show
df.loc[PD_indexes, 'sentence_text'].values

As shown above. PD often used as an abbreviation for parkinson's disease.
(also see https://www.ncbi.nlm.nih.gov/books/NBK536722/)

In [ ]:
# Replace " PD " with " parkinson disease ".
df['sentence_text'] = df['sentence_text'].str.replace("PD", "parkinson's disease", case= True)
# Show if correction was successful 
df.loc[PD_indexes, 'sentence_text'].values

Note that the code above is also used in `corpus_by_sentence2vectors.py` before embedding sentences.

In [ ]:
# Make all sentence_text lower case & remove punctuations
# https://www.geeksforgeeks.org/python/generating-word-cloud-python/
df['sentence_text'] = df['sentence_text'].str.lower().str.strip('.?!')

In [ ]:
# Search for sentences that include 'Parkinson'
sentences = df.loc[df['sentence_text'].str.findall("parkinson's disease").str.len() > 0, 'sentence_text'].values
sentences

Note that total number of sentences has now increased: 

5974: when searching only on "PD".           

6490: now together with "parkinson's disease".

## Frequency of header names

We would like to know what different headers are used throughout the different papers.

In [ ]:
# explore unique values
unique_values_per_column(df)

I did not expect to find "Western blot and tandem mass spectrometric analysis of protein-HNE adducts" in `head_name`. Let's check out if parsing went correct.

In [ ]:
df[df["head_name"] == "western blot and tandem mass spectrometric analysis of protein-hne adducts"] 

After checking the paper (PMID 17900545); "Western blot and tandem mass spectrometric analysis of protein-HNE adducts" is found to be a section from "Results". So these are parsed correctly.

I want to check if the longest headers are correct:

In [ ]:
# Display more characters in printed data frames
pd.set_option('display.max_colwidth', 110)

In [ ]:
# Select only unique 'paper_name' and 'head_name' combinations
selection = df[['paper_name','head_id','head_name']].drop_duplicates()

# Sort selection on 'head_name' string length
df.loc[selection['head_name'].str.len().sort_values(ascending=False).index, ['paper_name', 'head_name']]

I checked the longest two "head_name"s and they are correctly parsed. The shortest headers though, appear to be incorrectly parsed, so let's check that out as well.

In [ ]:
# Sort selection on 'head_name' string length
df.loc[selection['head_name'].str.len().sort_values().index].head(5)

All three papers (30248838, 29209747, and 23950519) I've checked contain type of parsing error: 30248838 header_id 15 is a sentence in the highlight section, 23950519 header_id 3 is the first letter of the sentence, for 29209747 header_id 37 I have no idea where the number 3 comes from, but the sentence starts on a new page.

In [ ]:
# turn all head_name s to lower case
df['head_name'] = df['head_name'].str.lower()

In [ ]:
# plot the distribution of the 'head_name' column
df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20).plot(kind='bar', figsize=(10, 6))

# annotate value above each bar
for i, v in enumerate(df.groupby(['head_name'])['paper_id'].nunique().sort_values(ascending=False).head(20)):
    plt.text(i, v + 0.5, str(v), ha='center', va='bottom')

plt.title('Distribution of head_name')
plt.ylabel('Number of unique paper_id')
# plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

The naming of headers appears to be inconsistent with different spellings: “conclusion” and “conclusions”, “acknowledgments” and “acknowledgements”. 

## Frequency words (wordcloud)

The following wordclouds will gain us insights if further parsing the papers went correctly.

Wordcloud python library works the best with English words (for auto removing stopwords) and removes "'s" from text. See [documentation](https://amueller.github.io/word_cloud/_modules/wordcloud/wordcloud.html#WordCloud).

**NOTE:** Keep in mind that the processed papers are already biased to (pesticides AND parkinson's disease).

### Combined Parkinson's disease & Pesticides

In [ ]:
# Number of unique papers in corpus
nu_total = df['paper_name'].nunique()

# Number of unique papers that have a sentence where both "pesticides" and "PD" are mentioned
nu_mentioned = df.loc[
    df['sentence_text'].str.contains(
        "|".join(["pesticides", "parkinson's disease"])
    ), 
    'paper_name'
].nunique()

# Test if each paper contain information about pesticides and PD. (If yes, then print should say `True`)
nu_total == nu_mentioned

In [ ]:
# All sentences
print(f"{df["sentence_text"].shape[0]} sentences.")
create_wordcloud(
    df["sentence_text"].values
)

In [ ]:
# Sentences with "pesticides", (AND????)"parkinson's disease"
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides", "parkinson's disease"],
    )
)

In [ ]:
# Sentences without "pesticides", "parkinson's disease"
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides", "parkinson's disease"],
        inverse= True
    )
)

### Parkinson's disease

In [ ]:
# with
create_wordcloud(
    get_sentences(
        df["sentence_text"], 
        ["parkinson's disease"]
    )
)

In [ ]:
# without
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["parkinson's disease"],
        inverse= True
    )
)

### Pesticides

In [ ]:
# with
create_wordcloud(
    get_sentences(
        df["sentence_text"], 
        ["pesticides"]
    )
)

In [ ]:
# without
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        ["pesticides"],
        inverse= True
    )
)

### Various other keywords

In [ ]:
# with Mitochondria
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "mitochondria"
    )
)

In [ ]:
# with Microglia
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "microglia"
    )
)

In [ ]:
# with Paraquat
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "paraquat"
    )
)

In [ ]:
# with Chlorpyrifos
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "chlorpyrifos"
    )
)

In [ ]:
# with Acetamiprid
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "acetamiprid"
    )
)

In [ ]:
# with Pendimethalin
create_wordcloud(
    get_sentences(
        df["sentence_text"],
        "pendimethalin"
    )
)

## Number of words per sentence

In order to choose the right model for determining sentence similarity, it is helpfull to know general properties (e.g. number of words per sentence). Some transformers can only handle a limited amount of words per sentence.

"A common value for BERT-based models are 512 tokens, which corresponds to about 300-400 words (for English)." https://www.sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html#input-sequence-length 

In [ ]:
# Load model of choice
model_st = SentenceTransformer('NeuML/pubmedbert-base-embeddings')

# Max number of tokens per sentence.
# Longer texts will be truncated to the first model.max_seq_length tokens
# https://www.sbert.net/examples/sentence_transformer/applications/computing-embeddings/README.html#input-sequence-length 
model_st.max_seq_length

The maximum number of words per sentence for our chosen model ('NeuML/pubmedbert-base-embeddings') is 512.

#### Check word count distribution of sentences in our corpus:

Let's find out about our number of words per sentence.

In [ ]:
# Create wordcount series
s_word_count = df['sentence_text'].str.split(' ').map(len)

In [ ]:
# Plot histogram
s_word_count.plot.hist(bins= 60)

plt.title("Words per sentence distribution")
plt.xlabel("Number of words")
plt.show()

In [ ]:
df_describe_not_norm(s_word_count)

It appears somewhat normal distributed with a heavy tail to the right.

Of the 30484 sentences, an median sentence contains 23 words, where the longest sentence is 262 words long.

To use 'Sentence-BERT' for spaCy the sentences need to be shorter than 128.

In [ ]:
# Sentences that are longer than 128 words.
print(f"There are {s_word_count[s_word_count >128].shape[0]} sentences longer than 128.")

I'm wondering what these long sentences are.

In [ ]:
# Get index of long sentences
i_long_sentences = s_word_count[s_word_count >128].index

In [ ]:
df.loc[i_long_sentences]

The long sentences appear in different sections throughout a paper.

## Number of sentences per paper

Number of sentences per paper:

In [ ]:
# Number of headers, paragraphs, and sentences per paper.
df.groupby('paper_id')[['head_id', 'paragraph_id', 'sentence_text']].nunique()

#### Number of sentences per paper globally described

In [ ]:
df.groupby('paper_id')[['sentence_text']].nunique().plot.hist(bins= 25)
plt.title("Distribution of total number of sentences per paper")
plt.xlabel('Number of sentences')
plt.show()

In [ ]:
df_describe_not_norm(df.groupby('paper_id')[['sentence_text']].nunique()['sentence_text'])

Of the 152 papers, an median sentence contain 182 sentences, where the longest paper is 1127 sentences long.

In [ ]:
df.groupby('paper_name')[['sentence_text']].nunique().sort_values(by= 'sentence_text', ascending=False)

The 1127 sentences in PMID 21626386 appears not to be a parsing error, for the paper contains 207 pages.